In [ ]:
from google.colab import drive
drive.mount('/content/drive') # Mount Google Drive to access datasets

In [ ]:
import pandas as pd # Import pandas for data manipulation
import numpy as np  # Import numpy for numerical operations

In [ ]:
# Load datasets from Google Drive
temperature_df = pd.read_csv("/content/drive/MyDrive/Dataset/tbltemperature.csv")  # Temperature data
neutrophils_df = pd.read_csv("/content/drive/MyDrive/Dataset/tblwbc.csv")           # Neutrophil counts (White Blood Cell data)
stool_df = pd.read_csv("/content/drive/MyDrive/Dataset/tblASVsamples.csv")         # Stool consistency data
allo_hct = pd.read_csv("/content/drive/MyDrive/Dataset/tblhctmeta.csv")            # Allogeneic Hematopoietic Cell Transplantation metadata

In [ ]:
temperature_df.info() # Display concise summary of the temperature DataFrame

In [ ]:
neutrophils_df.info() # Display concise summary of the neutrophils DataFrame

In [ ]:
stool_df.info() # Display concise summary of the stool DataFrame

In [ ]:
allo_hct.info() # Display concise summary of the allo_hct DataFrame

In [ ]:
# Convert 'PatientID' to string type across all DataFrames to ensure consistent merging
temperature_df['PatientID'] = temperature_df['PatientID'].astype(str)
neutrophils_df['PatientID'] = neutrophils_df['PatientID'].astype(str)
stool_df['PatientID'] = stool_df['PatientID'].astype(str)
allo_hct['PatientID'] = allo_hct['PatientID'].astype(str)

In [ ]:
# Filter out only the necessary columns from each DataFrame
temperature_df = temperature_df[['PatientID', 'DayRelativeToNearestHCT', 'MaxTemperature']]
# Filter for 'Neutrophils' and select relevant columns
neutrophils_df = neutrophils_df[neutrophils_df['BloodCellType'] == 'Neutrophils'][['PatientID', 'DayRelativeToNearestHCT', 'Value']]
stool_df = stool_df[['PatientID', 'DayRelativeToNearestHCT', 'Consistency']]
allo_hct = allo_hct[['PatientID', 'TimepointOfTransplant', 'Disease']]

In [ ]:
# Rename the 'Value' column to 'NeutrophilCount' in neutrophils_df for clarity and consistency
neutrophils_df.rename(columns={'Value': 'NeutrophilCount'}, inplace=True)

In [ ]:
# Merge datasets using 'PatientID' and 'DayRelativeToNearestHCT' as keys
# Use 'outer' join to keep all records from both DataFrames.
merged_df = pd.merge(allo_hct, temperature_df, on=['PatientID'], how='outer')
merged_df = pd.merge(merged_df, neutrophils_df, on=['PatientID', 'DayRelativeToNearestHCT'], how='outer')
merged_df = pd.merge(merged_df, stool_df, on=['PatientID', 'DayRelativeToNearestHCT'], how='outer')

In [ ]:
merged_df.head(5) # Display the first 5 rows of the merged DataFrame to inspect the data

In [ ]:
# Define the start and end days relative to allo-HSCT for a uniform timeline
min_day = -15  # Minimum day (e.g., 15 days before transplant)
max_day = 35   # Maximum day (e.g., 35 days after transplant)

In [ ]:
# Create a DataFrame containing all days within the defined timeline
all_days = pd.DataFrame(
    {'DayRelativeToNearestHCT': range(min_day, max_day + 1)})
# Get a unique list of all PatientIDs present in the merged dataset
patients = merged_df['PatientID'].unique()

In [ ]:
# Calculate and print the total number of unique patients
num_patients = patients.shape[0]  # or len(patients)
print(f"Total number of patients: {num_patients}")

In [ ]:
# Initialize an empty list to store processed data for each patient
aligned_data = []

In [ ]:
for patient in patients:
    # Filter the merged DataFrame to get data for the current patient
    patient_data = merged_df[merged_df['PatientID'] == patient]

    # Merge patient's data with the complete timeline ('all_days')
    # This ensures that each patient has a record for every day in the specified range,
    # filling in NaNs for days where no data was originally available.
    patient_timeline = all_days.merge(
        patient_data, on='DayRelativeToNearestHCT', how='left')
    patient_timeline['PatientID'] = patient  # Explicitly add PatientID to the timeline
    aligned_data.append(patient_timeline) # Add the patient's complete timeline to the list

In [ ]:
# Concatenate all individual patient timelines into a single DataFrame
# ignore_index=True resets the index of the resulting DataFrame
final_df = pd.concat(aligned_data, ignore_index=True)

In [ ]:
# Handle missing data using appropriate strategies:
# Forward-fill 'MaxTemperature': Propagate the last valid observation forward to next valid observation
final_df['MaxTemperature'] = final_df['MaxTemperature'].ffill()
# Forward-fill 'NeutrophilCount': Propagate the last valid observation forward
final_df['NeutrophilCount'] = final_df['NeutrophilCount'].ffill()
# Fill missing 'Consistency' values with 'Unknown'
final_df['Consistency'] = final_df['Consistency'].fillna('Unknown')

In [ ]:
final_df.shape # Display the dimensions (rows, columns) of the final processed DataFrame

In [ ]:
# Save the fully processed DataFrame to a CSV file
# index=False prevents pandas from writing the DataFrame index as a column in the CSV
final_df.to_csv("processed_dataset.csv", index=False)
print("Dataset processed and saved as 'processed_dataset.csv'")